# 01 — Data Fetch

Load the anonymized lead data and generate synthetic marketing spend.

**Steps**:
1. Load the clean lead CSV (anonymized in notebook 00)
2. Copy to `data/raw/` as the project's raw data source
3. Generate synthetic spend data for CAC/ROAS analysis

Everything is saved to `data/raw/`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
LEADS_DIR = RAW_DIR / "leads"
SPEND_DIR = RAW_DIR / "spend"

LEADS_DIR.mkdir(parents=True, exist_ok=True)
SPEND_DIR.mkdir(parents=True, exist_ok=True)

## 1 · Load Lead Data

In [2]:
leads = pd.read_csv(PROJECT_ROOT / "datasets" / "insurance_leadgen_data.csv")

print(f"Leads: {leads.shape[0]:,} rows × {leads.shape[1]} cols")
print(f"\nColumns: {list(leads.columns)}")
print(f"\nLead Status:")
print(leads["lead_status"].value_counts())
leads.head()

Leads: 7,485 rows × 13 cols

Columns: ['lead_id', 'lead_status', 'premium', 'age', 'gender', 'smoker', 'current_insurance', 'device_type', 'keyword', 'match_type', 'postcode', 'cover_for', 'verification_status']

Lead Status:
lead_status
Contacted     5968
Quoted         510
Sold           367
Invalid        326
No Contact     314
Name: count, dtype: int64


,lead_id,lead_status,premium,age,gender,smoker,current_insurance,device_type,keyword,match_type,postcode,cover_for,verification_status
0,90809057,Contacted,0.0,31,Female,Yes,no,Desktop,private health insurance belfast,Exact,Bt13,Self,NaN
1,35622726,Contacted,0.0,35,Female,No,no,Smartphone,private medical insurance,Phrase,LU7,Self,NaN
2,29156003,Contacted,0.0,46,Male,No,yes private,Smartphone,private health insurance,Broad,E14,Self,NaN
3,26436791,No Contact,0.0,53,Female,No,no,Desktop,bupa health insurance,Exact,W2,Self + Partner,NaN
4,80270348,Contacted,0.0,35,Male,No,no,Smartphone,private healthcare prices,Exact,BH23,Self,NaN


In [3]:
# Copy to raw data folder
leads.to_csv(LEADS_DIR / "insurance_leads.csv", index=False)
print(f"Saved → data/raw/leads/insurance_leads.csv ({len(leads):,} rows)")

Saved → data/raw/leads/insurance_leads.csv (7,485 rows)


## 2 · Synthetic Lead Spend

We don't have real spend data, so we generate realistic marketing spend:
- Weekly granularity, keyed on `keyword_group × device × match_type`
- This lets us calculate CAC, ROAS, and cost-per-lead in later notebooks

In [4]:
np.random.seed(42)

# Keyword grouping (for spend generation)
def classify_keyword(kw):
    kw = str(kw).lower()
    if any(w in kw for w in ["health", "medical", "hospital"]):        return "health_insurance"
    elif any(w in kw for w in ["life", "death", "funeral"]):           return "life_insurance"
    elif any(w in kw for w in ["family", "couple", "partner"]):        return "family_cover"
    elif any(w in kw for w in ["cheap", "affordable", "budget"]):      return "budget_insurance"
    elif any(w in kw for w in ["private", "premium", "bupa", "axa"]): return "private_insurance"
    elif any(w in kw for w in ["compare", "quote", "best"]):           return "comparison_shopping"
    else: return "general_insurance"

leads["keyword_group"] = leads["keyword"].apply(classify_keyword)

# Generate weekly spend
weeks = pd.date_range("2025-01-06", "2025-12-29", freq="W-MON")
keyword_groups = leads["keyword_group"].unique().tolist()
device_types = ["Desktop", "Smartphone", "Tablet"]
match_types = ["Exact", "Phrase", "Broad"]
cpc_ranges = {"Exact": (2.5, 6.0), "Phrase": (1.5, 4.0), "Broad": (0.8, 2.5)}

rows = []
for week in weeks:
    for kg in keyword_groups:
        for device in device_types:
            for match in match_types:
                device_mult = {"Desktop": 1.3, "Smartphone": 1.0, "Tablet": 0.5}[device]
                clicks = int(np.random.randint(10, 80) * device_mult)
                low, high = cpc_ranges[match]
                cpc = round(np.random.uniform(low, high), 2)
                rows.append({
                    "week_start": week, "keyword_group": kg, "device_type": device,
                    "match_type": match, "impressions": int(clicks * np.random.uniform(8, 25)),
                    "clicks": clicks, "spend": round(clicks * cpc, 2), "avg_cpc": cpc,
                })

lead_spend = pd.DataFrame(rows)
print(f"Lead spend: {lead_spend.shape[0]:,} rows")
print(f"Total spend: £{lead_spend['spend'].sum():,.0f}")
print(f"Date range: {lead_spend['week_start'].min().date()} → {lead_spend['week_start'].max().date()}")

Lead spend: 1,404 rows
Total spend: £164,308
Date range: 2025-01-06 → 2025-12-29


In [5]:
# Save spend
lead_spend.to_csv(SPEND_DIR / "lead_spend.csv", index=False)
print(f"Saved → data/raw/spend/lead_spend.csv ({lead_spend.shape[0]:,} rows)")

Saved → data/raw/spend/lead_spend.csv (1,404 rows)


## Summary

In [6]:
print("RAW DATA SUMMARY")
print("=" * 45)
print(f"Insurance leads: {len(leads):>8,} rows")
print(f"Lead spend:      {lead_spend.shape[0]:>8,} rows (synthetic)")
print(f"\nAll saved to data/raw/")

RAW DATA SUMMARY
Insurance leads:    7,485 rows
Lead spend:         1,404 rows (synthetic)

All saved to data/raw/
